# 数据可视化（Data Visualization）— Phase 1: EDA 探索性数据分析（Exploratory Data Analysis）

用 **Plotly**（交互式图表库，interactive plotting library）认识我们的电价数据（`V2.5_15min_features.csv`）。

本 notebook 共 **6 张交互图**，每张图都配一段讲解，帮助你回答：

- 电价整体长什么样？（趋势 / 分布）
- 一天中什么时候最贵？（小时模式）
- 周末 / 季节有影响吗？（日历模式）
- 天气和价格有关系吗？（相关性）

> 用法提示：鼠标**悬停**看精确数值，可**缩放 / 拖动**，点击图例可隐藏某条线。

## 0. 环境准备（Setup）

用到的库（libraries）：
- **pandas**：读数据、处理表格
- **numpy**：数值计算
- **plotly.express (px)**：快速画交互式图表的接口（interface）
- **plotly.graph_objects (go)**：更底层、更灵活的绘图接口

> 如果报错 `ModuleNotFoundError: No module named 'plotly'`，说明没装，在终端运行：
> `pip install plotly`

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print('Libraries ready ✅')

Libraries ready ✅


In [2]:
# 读取特征数据表（15 分钟粒度，2023-2025）
df = pd.read_csv('../data/convertData/V2.5_15min_features.csv')
print('原始形状（rows, columns）:', df.shape)

# 时区处理：数据里有混合时区（冬+02:00 / 夏+03:00），先统一转 UTC 再转赫尔辛基
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)

# 从时间列再提取几个画图要用的列
df['date'] = pd.to_datetime(df['datetime'].dt.date)
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek   # 0=周一, 6=周日
df['month'] = df['datetime'].dt.month

print('时间范围:', df['datetime'].min(), '→', df['datetime'].max())
print('总行数:', len(df))
df.head()

原始形状（rows, columns）: (105216, 51)
时间范围: 2023-01-01 00:00:00+02:00 → 2025-12-31 23:45:00+02:00
总行数: 105216


,datetime,price,temp,wind_speed,wind_direction_deg,wind_dir_sin,wind_dir_cos,hour,minute,day_of_week,...,price_rolling_std_24h,price_rolling_min_24h,price_rolling_max_24h,price_rolling_mean_7d,temp_rolling_mean_1h,HDD,wind_power_proxy,temp_lag_4,temp_lag_96,date
0,2023-01-01 00:00:00+02:00,4.8400,4.45,8.45,215.35,-0.578569,-0.815633,0,0,6,...,NaN,NaN,NaN,NaN,NaN,12.55,603.351125,NaN,NaN,2023-01-01
1,2023-01-01 00:15:00+02:00,4.1325,4.50,9.00,213.10,-0.546102,-0.837719,0,15,6,...,NaN,NaN,NaN,NaN,NaN,12.50,729.000000,NaN,NaN,2023-01-01
2,2023-01-01 00:30:00+02:00,3.4250,4.45,8.55,211.25,-0.518650,-0.854708,0,30,6,...,NaN,NaN,NaN,NaN,NaN,12.55,625.026375,NaN,NaN,2023-01-01
3,2023-01-01 00:45:00+02:00,2.7175,4.30,8.10,209.70,-0.495459,-0.868632,0,45,6,...,NaN,NaN,NaN,NaN,NaN,12.70,531.441000,NaN,NaN,2023-01-01
4,2023-01-01 01:00:00+02:00,2.0100,4.30,8.10,211.85,-0.527451,-0.849036,1,0,6,...,NaN,NaN,NaN,NaN,4.425,12.70,531.441000,4.45,NaN,2023-01-01


## 1.1 电价随时间变化（Electricity Price Over Time）

**这是什么图**：折线图（line chart）。横轴 = 时间，纵轴 = 价格（EUR/MWh）。

**怎么看**：
- 鼠标**悬停**在曲线上，能看到精确的时间和价格
- 可以**拖动缩放**、双击还原
- 注意看：冬天比夏天贵吗？有没有突然的尖峰？

**为什么重要**：这是认识数据的第一步——看到整体形状后，你才知道后面该注意什么。

> 小技巧：105,216 行太多，交互图会卡。这里随机抽样（sample）2 万行，既快又能看到整体趋势。

In [3]:
# 随机抽 2 万行，保证交互图流畅
sample = df.sample(n=20000, random_state=42).sort_values('datetime')

fig = px.line(sample, x='datetime', y='price',
              title='1.1 电价随时间变化（Electricity Price Over Time）',
              labels={'datetime': '时间 Time', 'price': '价格 Price (EUR/MWh)'})
fig.show()

## 1.2 价格分布（Price Distribution）

**这是什么图**：直方图（histogram）。把价格分成很多小格子，数每个格子里有多少条数据。

**怎么看**：
- 看横轴的**集中区域**：大部分价格集中在哪个区间？（可能 0~50 EUR/MWh？）
- 看**右尾巴**：有没有少数极端高价？（尖峰事件）
- 看有没有**负价格**：风电多的时候价格可能为负

**为什么重要**：了解"典型价格长什么样"，以后判断模型预测合不合理就有参照。

In [4]:
fig = px.histogram(df, x='price', nbins=120,
                   title='1.2 价格分布（Price Distribution）',
                   labels={'price': '价格 Price (EUR/MWh)', 'count': '数量 Count'})
fig.update_layout(showlegend=False)
fig.show()

## 1.3 一天中各小时的价格（Price by Hour of Day）

**这是什么图**：箱线图（box plot）。每个小时一列，显示该小时价格的中位数（median，中位数）、四分位范围、异常点。

**怎么看**：
- 中间的**横线** = 中位数
- 看哪个小时中位数最高 → 那就是**用电高峰**
- 一般早晚各有一个高峰（大家起床 / 回家用电）

**为什么重要**：电价有明显的日内（intraday，日内）规律，这是后面做特征（如 `is_peak_hour`）的依据。

In [5]:
# df 里已有 hour 列（时间特征），直接用
fig = px.box(df, x='hour', y='price',
             title='1.3 一天中各小时的价格分布（Price by Hour of Day）',
             labels={'hour': '小时 Hour', 'price': '价格 Price (EUR/MWh)'})
fig.show()

## 1.4 星期 & 月份模式（Weekday & Seasonal Pattern）

**这是什么图**：柱状图（bar chart）。每根柱子 = 一组时间段（某星期 / 某月份）的平均价格。

**怎么看**：
- 星期图：周末平均价是不是比工作日低？（工厂停产、需求下降）
- 月份图：冬天（12-2月）是不是比夏天贵？（供暖需求）

**为什么重要**：证明"日历特征（calendar features）"是有意义的——这也解释了为什么模型里加了 `day_of_week`、`season` 这些特征。

In [6]:
# 1.4a 按星期：平均价格
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df['day_name'] = df['day_of_week'].map(dict(enumerate(day_names)))
avg_day = df.groupby('day_name', as_index=False)['price'].mean()

fig1 = px.bar(avg_day, x='day_name', y='price',
              category_orders={'day_name': day_names},
              title='1.4a 各星期平均电价（Average Price by Weekday）',
              labels={'day_name': '星期 Weekday', 'price': '平均价格 Avg Price (EUR/MWh)'})
fig1.show()

# 1.4b 按月份：平均价格
avg_month = df.groupby('month', as_index=False)['price'].mean()
fig2 = px.bar(avg_month, x='month', y='price',
              title='1.4b 各月份平均电价（Average Price by Month）',
              labels={'month': '月份 Month', 'price': '平均价格 Avg Price (EUR/MWh)'})
fig2.show()

## 1.5 天气 vs 价格（Weather vs Price）

**这是什么图**：散点图（scatter plot）。每个点 = 一个 15 分钟时刻，横轴 = 天气，纵轴 = 价格。

**怎么看**：
- 温度 vs 价格：温度低的时候，价格是不是整体更高？（负相关，negative correlation）
- 风速 vs 价格：风速高的时候，风电多，价格是不是更低？

**为什么重要**：这是"为什么模型要用天气做特征"的直接证据。

> 点太多会糊在一起，所以设置 `opacity=0.3`（半透明）和抽样 2 万行。

In [7]:
sample = df.sample(n=20000, random_state=42)

fig1 = px.scatter(sample, x='temp', y='price', opacity=0.3,
                  title='1.5a 温度 vs 电价（Temperature vs Price）',
                  labels={'temp': '温度 Temperature (°C)', 'price': '价格 Price (EUR/MWh)'})
fig1.show()

fig2 = px.scatter(sample, x='wind_speed', y='price', opacity=0.3,
                  title='1.5b 风速 vs 电价（Wind Speed vs Price）',
                  labels={'wind_speed': '风速 Wind Speed (m/s)', 'price': '价格 Price (EUR/MWh)'})
fig2.show()

## 1.6 相关性热力图（Correlation Heatmap）

**这是什么图**：热力图（heatmap）。每个格子 = 两个变量的**相关系数**（correlation coefficient），取值 -1 到 +1。

**怎么看**：
- **+1（红）**：两个变量一起变大（正相关，positive correlation）
- **-1（蓝）**：一个变大另一个变小（负相关）
- **0（白）**：没关系

**重点看**：哪一列和 `price` 的格子颜色最深？那说明它和价格关系最密切。

**为什么重要**：相关性高的特征往往对预测有用；相关性极高的两个特征之间可能冗余（redundant）。

In [8]:
# 只挑数值列 + 和价格可能相关的特征，避免图太挤
cols = ['price', 'temp', 'wind_speed', 'HDD', 'wind_power_proxy',
        'price_lag_1', 'price_lag_96', 'price_rolling_mean_1h',
        'hour', 'day_of_week']
corr = df[cols].corr()

fig = px.imshow(corr, text_auto=True, aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='1.6 相关性热力图（Correlation Heatmap）')
fig.show()